# Analyze inference predictions

Add readable labels and exact inference run IDs to `RUNS`. The remaining cells load
their manifest-declared predictions, join them to qrels, and build tidy pandas
DataFrames ready for project-specific plots.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from IPython.display import display
except ImportError:  # pragma: no cover - display convenience outside notebooks
    display = print

In [ ]:
PROJECT_ROOT = Path(__file__).resolve().parents[2]
RUNS_DIR = PROJECT_ROOT / "artifacts" / "runs" / "inference"
QRELS_PATH = (
    PROJECT_ROOT / "data" / "processed" / "beir" / "{{ cookiecutter.beir_dataset }}" / "qrels.jsonl"
)

# Map readable labels used in plots to exact inference run IDs.
RUNS: dict[str, str] = {
    # "baseline": "{{ cookiecutter.project_slug }}-baseline-YYYYMMDD-HHMMSS",
    # "treatment": "{{ cookiecutter.project_slug }}-treatment-YYYYMMDD-HHMMSS",
}

In [ ]:
PREDICTION_COLUMNS = [
    "run_label",
    "run_id",
    "query_id",
    "query",
    "rank",
    "document_id",
    "evaluation_document_id",
    "score",
    "title",
    "content",
    "meta",
]
QUERY_COLUMNS = ["run_label", "run_id", "query_id", "query"]


def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def predictions_path_for_run(
    run_id: str, *, runs_dir: Path = RUNS_DIR, project_root: Path = PROJECT_ROOT
) -> Path:
    manifest_path = runs_dir / run_id / "manifest.json"
    manifest = read_json(manifest_path)
    configured_path = Path(manifest["artifacts"]["predictions"])
    predictions_path = (
        configured_path if configured_path.is_absolute() else project_root / configured_path
    )
    if not predictions_path.is_file():
        raise FileNotFoundError(f"Predictions do not exist: {predictions_path}")
    return predictions_path


def load_qrels(path: Path) -> pd.DataFrame:
    qrels = pd.DataFrame(read_jsonl(path), columns=["query_id", "document_id", "relevance"])
    qrels["query_id"] = qrels["query_id"].astype(str)
    qrels["document_id"] = qrels["document_id"].astype(str)
    qrels["relevance"] = pd.to_numeric(qrels["relevance"], errors="raise").astype(int)
    return qrels.groupby(["query_id", "document_id"], as_index=False, sort=False).agg(
        relevance=("relevance", "max")
    )


def load_run(
    run_label: str,
    run_id: str,
    *,
    runs_dir: Path = RUNS_DIR,
    project_root: Path = PROJECT_ROOT,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    payload = read_json(
        predictions_path_for_run(run_id, runs_dir=runs_dir, project_root=project_root)
    )
    prediction_rows: list[dict[str, Any]] = []
    query_rows: list[dict[str, str | None]] = []

    for query_id, query_payload in payload.items():
        query = query_payload.get("query")
        query_rows.append(
            {"run_label": run_label, "run_id": run_id, "query_id": str(query_id), "query": query}
        )
        for rank, (document_id, document) in enumerate(
            query_payload.get("documents", {}).items(), start=1
        ):
            meta = dict(document.get("meta") or {})
            prediction_rows.append(
                {
                    "run_label": run_label,
                    "run_id": run_id,
                    "query_id": str(query_id),
                    "query": query,
                    "rank": rank,
                    "document_id": str(document_id),
                    "evaluation_document_id": str(meta.get("source_document_id") or document_id),
                    "score": document.get("score"),
                    "title": meta.get("title"),
                    "content": document.get("content"),
                    "meta": meta,
                }
            )

    return (
        pd.DataFrame(prediction_rows, columns=PREDICTION_COLUMNS),
        pd.DataFrame(query_rows, columns=QUERY_COLUMNS),
    )


def build_analysis_frames(
    runs: dict[str, str],
    qrels_path: Path,
    *,
    runs_dir: Path = RUNS_DIR,
    project_root: Path = PROJECT_ROOT,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    qrels = load_qrels(qrels_path)
    loaded_runs = [
        load_run(run_label, run_id, runs_dir=runs_dir, project_root=project_root)
        for run_label, run_id in runs.items()
    ]
    predictions = pd.concat([item[0] for item in loaded_runs], ignore_index=True)
    queries = pd.concat([item[1] for item in loaded_runs], ignore_index=True)

    qrel_join = qrels.rename(columns={"document_id": "evaluation_document_id"})
    predictions = predictions.merge(
        qrel_join,
        how="left",
        on=["query_id", "evaluation_document_id"],
        validate="many_to_one",
        indicator=True,
    )
    predictions["is_judged"] = predictions.pop("_merge").eq("both")
    predictions["relevance"] = predictions["relevance"].fillna(0).astype(int)
    predictions["is_relevant"] = predictions["relevance"].gt(0)
    predictions["score"] = pd.to_numeric(predictions["score"], errors="coerce")

    relevant_counts = (
        qrels.loc[qrels["relevance"].gt(0)]
        .groupby("query_id")["document_id"]
        .nunique()
        .rename("relevant_document_count")
    )
    predictions = predictions.join(relevant_counts, on="query_id")
    predictions["relevant_document_count"] = (
        predictions["relevant_document_count"].fillna(0).astype(int)
    )

    # Match evaluation semantics by counting a source document only once when predictions are chunks.
    collapsed = predictions.sort_values("rank").drop_duplicates(
        ["run_label", "query_id", "evaluation_document_id"]
    )
    collapsed["relevant_rank"] = collapsed["rank"].where(collapsed["is_relevant"])
    retrieved = collapsed.groupby(["run_label", "run_id", "query_id"], as_index=False).agg(
        retrieved_document_count=("evaluation_document_id", "nunique"),
        retrieved_relevant_count=("is_relevant", "sum"),
        first_relevant_rank=("relevant_rank", "min"),
        run_depth=("rank", "max"),
    )

    query_summary = queries.merge(
        retrieved, how="left", on=["run_label", "run_id", "query_id"], validate="one_to_one"
    ).join(relevant_counts, on="query_id")
    count_columns = [
        "retrieved_document_count",
        "retrieved_relevant_count",
        "run_depth",
        "relevant_document_count",
    ]
    query_summary[count_columns] = query_summary[count_columns].fillna(0).astype(int)
    query_summary["first_relevant_rank"] = query_summary["first_relevant_rank"].astype("Int64")
    query_summary["reciprocal_rank"] = (
        1.0 / query_summary["first_relevant_rank"].astype("Float64")
    ).fillna(0.0)
    denominator = query_summary["relevant_document_count"].where(
        query_summary["relevant_document_count"].gt(0)
    )
    query_summary["recall_at_run_depth"] = (
        query_summary["retrieved_relevant_count"].div(denominator).fillna(0.0)
    )
    query_summary["query_character_count"] = query_summary["query"].fillna("").str.len()
    query_summary["query_word_count"] = query_summary["query"].fillna("").str.split().str.len()

    run_order = {label: position for position, label in enumerate(runs)}
    predictions["_run_order"] = predictions["run_label"].map(run_order)
    query_summary["_run_order"] = query_summary["run_label"].map(run_order)
    predictions = predictions.sort_values(["_run_order", "query_id", "rank"]).drop(
        columns="_run_order"
    )
    query_summary = query_summary.sort_values(["_run_order", "query_id"]).drop(columns="_run_order")
    return predictions.reset_index(drop=True), query_summary.reset_index(drop=True), qrels

In [ ]:
if RUNS:
    predictions_df, query_summary_df, qrels_df = build_analysis_frames(RUNS, QRELS_PATH)
    display(predictions_df.head())
    display(query_summary_df.head())
else:
    predictions_df = pd.DataFrame()
    query_summary_df = pd.DataFrame()
    qrels_df = pd.DataFrame()
    print("Add exact inference run IDs to RUNS, then rerun this cell.")

In [ ]:
# Add project-specific plotting code below. For paired run comparisons, a useful start is:
# plot_data = query_summary_df.pivot(
#     index="query_id", columns="run_label", values="reciprocal_rank"
# )
